# Radiology Model

This notebook trains and evaluates the `radiology` text classification model. It reads reports, applies keyword labeling rules, runs classical ML benchmarks, and demonstrates the LLM structured radiology extractor option.

In [ ]:
# Imports and project path setup
from pathlib import Path
import sys, json
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Set ROOT path relative to project structure
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.diagnostic_models import MODEL_SPECS, build_io_schema, candidate_pipelines, evaluate_pipeline, predict_for_backend, train_model


In [ ]:
# 1. Load Dataset
csv_path = ROOT / 'data' / 'real_processed' / 'iu_xray_reports_real.csv'
df = pd.read_csv(csv_path)
display(df.head())
print('Shape:', df.shape)


In [ ]:
# 2. Cleaning and Labeling
spec = MODEL_SPECS['radiology']
df = df.dropna(subset=['report_text']).copy()

def label_radiology(text):
    t = str(text).lower()
    urgent_terms = ['pneumothorax', 'pulmonary edema', 'acute', 'collapse', 'mediastinal shift', 'large pleural effusion', 'consolidation']
    follow_terms = ['opacity', 'nodule', 'mass', 'effusion', 'atelectasis', 'cardiomegaly', 'emphysema', 'fibrosis', 'granuloma', 'hernia', 'infiltrate']
    negated_urgent = ['no pneumothorax', 'without pneumothorax', 'no acute']
    if any(term in t for term in urgent_terms) and not any(term in t for term in negated_urgent):
        return 'urgent_review'
    if any(term in t for term in follow_terms):
        return 'follow_up'
    return 'routine'

df['urgency'] = df['report_text'].map(label_radiology)
print(df['urgency'].value_counts(normalize=True).round(3))


In [ ]:
# 3. Train / Validation / Test Split
X = df['report_text']
y = df['urgency']
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)
print(f'Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}')


In [ ]:
# 4. Preprocessing + Model Selection (Classical TF-IDF baseline)
candidate_results = {}
fitted = {}
for candidate_name, pipeline in candidate_pipelines(spec).items():
    pipeline.fit(X_train, y_train)
    fitted[candidate_name] = pipeline
    candidate_results[candidate_name] = evaluate_pipeline(pipeline, X_val, y_val, spec)

display(pd.DataFrame(candidate_results).T.sort_values(['positive_recall', 'macro_f1'], ascending=False))
best_name = sorted(candidate_results, key=lambda k: (candidate_results[k]['positive_recall'], candidate_results[k]['macro_f1'], candidate_results[k]['accuracy']), reverse=True)[0]
pipeline = fitted[best_name]
print('Best classical model:', best_name)


In [ ]:
# 5. Classical Evaluation Metrics
for split_name, X_split, y_split in [('validation', X_val, y_val), ('test', X_test, y_test)]:
    preds = pipeline.predict(X_split)
    print('\n' + split_name.upper())
    print(classification_report(y_split, preds, zero_division=0))
    print(confusion_matrix(y_split, preds, labels=pipeline.classes_))


In [ ]:
# 6. Save Classical Model (Optional)
joblib.dump(pipeline, ROOT / 'models' / 'radiology_pipeline.joblib')
print('Classical baseline model saved successfully.')


## 7. Advanced: LLM Radiology Extractor

For production, we use the LLM Radiology Extractor which provides clinical structured data and patient-friendly explanations in Egyptian Arabic. Let's test it:

In [ ]:
from src.radiology_extractor import extract_radiology_triage

sample_report = "Chest radiograph: Heart size is normal. No focal air space consolidation or pleural effusion. There is a right-sided apical pneumothorax."
extracted_data = extract_radiology_triage(sample_report)
print(json.dumps(extracted_data, indent=2, ensure_ascii=False))
